# 10 - Oxygen case study (Case Study 2)

Loads the public daily dissolved-oxygen target table and benchmark results for
Tongoy Balsa (BTGOXD2 sensor, mg/L). This mirrors the chlorophyll workflow in
notebooks 01-08, applied to a second sensor, per the "adapting the workflow to
a new sensor" pattern in notebook 09. Fully executable on the public data
included in this repository.

In [ ]:
import pandas as pd

DATA_DIR = "../data_public/oxygen"
RESULTS_DIR = "../results_public/oxygen"

target_df = pd.read_csv(f"{DATA_DIR}/oxygen_daily_target.csv", parse_dates=["date"])
target_df = target_df.set_index("date").sort_index()
target_df.head()


## Coverage summary

Eligibility mirrors the chlorophyll rule (>=18 valid hours/day required for a
trustworthy daily mean). Column `eligible_ge_18` is the eligibility flag used
throughout the oxygen benchmark.

In [ ]:
n_days = len(target_df)
n_eligible = int(target_df["eligible_ge_18"].sum())
coverage_summary = pd.Series({
    "n_days_total": n_days,
    "n_days_eligible": n_eligible,
    "pct_eligible": round(100 * n_eligible / n_days, 1),
    "n_days_missing_or_ineligible": n_days - n_eligible,
    "date_min": target_df.index.min().date(),
    "date_max": target_df.index.max().date(),
})
coverage_summary


## Real gap structure

Naturally occurring missing-data periods, grouped by length class. Real gaps
have no withheld ground truth -- see `docs/evidence_hierarchy.md`.

In [ ]:
real_gaps = pd.read_csv(f"{DATA_DIR}/oxygen_real_gap_inventory_by_class.csv")
real_gaps


## Artificial-gap validation pool

Withheld-truth validation gaps used to score every method. The **primary**
support is L=1-30 days (406 gaps) -- a more compact range than chlorophyll's
L=1-60, reflecting the shorter/denser missingness structure of the oxygen
record. The pool also carries 6 **exploratory extended** gaps at L=45/60/90/120,
not used in any benchmark number below (too few gaps at each length for a
meaningful comparison). See the `support_role` column and
`docs/data_dictionary.md`.

In [ ]:
gap_pool = pd.read_csv(f"{DATA_DIR}/oxygen_validation_gaps.csv")
print(f"Total artificial gaps: {len(gap_pool)}")
if "support_role" in gap_pool.columns:
    print(gap_pool["support_role"].value_counts())
gap_pool["gap_length"].value_counts().sort_index() if "gap_length" in gap_pool.columns else gap_pool.head()


## Benchmark summary

Mean absolute error (mg/L) by gap length for each method family, on the
oxygen artificial-gap validation pool. `TS-ICL physical covariates` (SST +
wind + solar + current) is the only method that clearly improves on
linear interpolation in the pooled benchmark -- 8.0% lower MAE, 95% CI
[4.5%, 11.4%].

In [ ]:
bench = pd.read_csv(f"{RESULTS_DIR}/oxygen_benchmark_by_length.csv")
bench_pivot = bench.pivot_table(
    index="gap_length", columns="method_label", values="mae_gapweighted"
)
bench_pivot


In [ ]:
deltas = pd.read_csv(f"{RESULTS_DIR}/oxygen_paired_deltas_vs_tsicl_physical_covariates.csv")
deltas[["comparator_label", "n_gaps", "delta_tsicl_minus_comparator_mae", "ci_lo", "ci_hi", "ci_excludes_zero"]]


## Local-BTG arm: diagnostic-only, never a core predictor result

`local_btg_temp_pressure_diagnostic` (water temperature + pressure, BTGTA/BTGPA)
is measured on the *same physical buoy* as the oxygen sensor (BTGOXD2) itself.
Its availability may covary with oxygen sensor outages for reasons unrelated to
any genuine physical relationship -- a day with no oxygen reading is more
likely to also have no BTGTA/BTGPA reading. For this reason this arm is
restricted to diagnostic/appendix use throughout this package
(`experiments.oxygen.benchmark_contract.LOCAL_BTG_TEMP_PRESSURE_ROLE`) and must
never be presented as a reliably available predictor during an oxygen-sensor
failure, even though it does show a real, CI-significant TS-ICL improvement in
isolation (+5.3%, see `oxygen_tsicl_audit.md` F.1-F.2).</cell>


## Distribution-tail limitation

TS-ICL's pooled improvement is not uniform across the oxygen distribution: it
is worse than interpolation in both tails at the point-estimate level, most
severely in sustained high-oxygen runs (small n, interpret cautiously). See
`figures/oxygen/figure_oxygen_tail_diagnostics.pdf` and Section 5.2 of the
report for the full discussion.

## Reproducing and extending this case study

Every table above is the **frozen, released result** -- the authoritative
number for its method, not regenerated by this notebook. The full pipeline
that could regenerate them (`experiments/oxygen/`: `benchmark_contract.py`,
`feature_registry.py`, `classical_models.py`, `tsicl_models.py`,
`tail_diagnostics.py`, `run_oxygen_benchmark.py`) is published and tested,
at three reproducibility levels:

- **QUICK** (minutes, no live TS-ICL checkpoint needed): the test suite,
  this notebook, and frozen-result inspection --
  `python -m experiments.oxygen.run_oxygen_benchmark --mode frozen`.
- **BOUNDED** (well under an hour): the classical Model-0/GP comparators on
  the complete 406-gap primary support (`--mode classical`, ~30s, reproduces
  the frozen Model-0/GP numbers in the Benchmark summary cell above almost
  exactly) and a deterministic stratified live TS-ICL subset of at most 20
  gaps (`--mode tsicl-bounded`, at most 60 real inference calls) that
  validates checkpoint loading, raw mg/L input/output, tensor shapes, and
  leakage safety -- **not** a new headline performance number, since the
  subset is small and length-biased.
- **EXPENSIVE, optional**: the complete TS-ICL grid (5 audited-original arms
  x 2 context modes + 4 exploratory family-ablation arms, all 406 primary
  gaps) is not re-run by this package -- the frozen tables above are the
  authoritative complete result, exactly as with chlorophyll's own full
  covariate dissection.

- Figures are sourced directly from the published report
  (`manuscript/report/`), not regenerated by `scripts/generate_public_figures.py`
  -- unlike the chlorophyll figures 1-5, the oxygen figures use report-specific
  paneling and are reproduced here as static assets (`figures/oxygen/`).
- See `docs/methodology/` for the shared validation protocol and model-family
  definitions that apply to both case studies, and
  `docs/methodology/oxygen_usage.md` for the oxygen-specific TS-ICL adaptation.

## Notes

- These figures/tables are sourced directly from the published report
  (`manuscript/report/`), not regenerated by
  `scripts/generate_public_figures.py` -- unlike the chlorophyll figures 1-5,
  the oxygen figures use report-specific paneling and are reproduced here as
  static assets (`figures/oxygen/`).
- See `docs/methodology/` for the shared validation protocol and model-family
  definitions that apply to both case studies.